# Feature Selection Strategy Implementation

**Pipeline order:** run this notebook first, then `baseline.ipynb` (optional), then any modeling notebook, then `modeling_compare_all.ipynb` to compare runs.

### Covered Workflow
- Stage 1: Broad univariate filtering (500 -> ~80)
- Stage 2: Multicollinearity and L1 regularization (80 -> ~25)
- Stage 3: CV drop-column ranking of Stage 2 features (business scorer, no full-data RFECV)

### Included Components
- Data loading and sanity checks
- Custom business scorer aligned with project metric
- Stage-by-stage feature reduction and validation
- Drop-column CV ranking of Stage 2 features
- Export of the Stage 2 ranking artifact consumed by the modeling notebook

## Key Parameters

| Parameter | Default | Description |
|---|---|---|
| custom_scorer | Required | Scoring function: scorer(estimator, X, y) -> float |
| stage1_n_features | 80 | Target features after Stage 1 |
| stage2_n_features | None | Optional strict target after Stage 2 |
| stage2_n_features_band | (20, 30) | Preferred approximate Stage 2 band (data-driven final count) |
| correlation_threshold | 0.85 | Correlation pruning threshold |
| vif_threshold | 5.0 | VIF filtering threshold |
| variance_threshold | 0.01 | Variance threshold (Stage 1) |
| ranking_cv_folds | 5 | CV folds for drop-column business ranking |

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cost_effective.dataset import (
    MulticollinearityFilter,
    UnivariateFeatureFilter,
    find_project_root,
    get_classifier,
    load_training_data,
)
from cost_effective.models import (
    evaluate_feature_sets,
    rank_features_drop_column_cv,
)

np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data"
OUTPUTS_PATH = PROJECT_ROOT / "outputs"

In [3]:
X_all, y = load_training_data(DATA_PATH)

print(f"✓ Loaded training data: {X_all.shape}")
print(f"  Class distribution: {y.value_counts().to_dict()}")

✓ Loaded training data: (5000, 500)
  Class distribution: {0: 2512, 1: 2488}


In [4]:
# Stage 1 and Stage 2
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*")
    warnings.filterwarnings("ignore", message=".*Inconsistent values: penalty=l1.*")

    print("=" * 70)
    print("STAGE 1: UNIVARIATE FILTERING (500 → 80)")
    print("=" * 70 + "\n")

    stage1_filter = UnivariateFeatureFilter(variance_threshold=0.01)
    X_stage1, stage1_features = stage1_filter.fit_transform(X_all, y, n_features=80)

    print()
    stage2_filter = MulticollinearityFilter(
        correlation_threshold=0.85,
        vif_threshold=5.0,
        random_state=42,
    )
    X_stage2, stage2_features = stage2_filter.fit_transform(
        X_stage1,
        y,
        n_features_target=None,
        n_features_band=(20, 30),
    )

print(f"✓ Stage 1 output: {len(stage1_features)} features")
print(f"✓ Stage 2 output: {len(stage2_features)} features")

STAGE 1: UNIVARIATE FILTERING (500 → 80)

[Stage 1a] Variance Threshold: 500 → 500 features
[Stage 1b] Mutual Information computed for 500 features
  Top 10 MI scores:
     feature  mi_score
10    var_10  0.029785
456  var_456  0.021367
470  var_470  0.020561
312  var_312  0.018987
159  var_159  0.018163
415  var_415  0.018163
175  var_175  0.018041
254  var_254  0.017258
4      var_4  0.017163
379  var_379  0.017061

[Stage 1c] LightGBM Importance computed
  Top 10 LGBM Importances:
     feature  lgbm_importance
214  var_214               94
379  var_379               94
341  var_341               81
254  var_254               73
190  var_190               71
116  var_116               67
159  var_159               64
226  var_226               61
389  var_389               60
482  var_482               58

[Stage 1 Output] Selected 80 features out of 500

STAGE 2: MULTICOLLINEARITY & L1 REGULARIZATION FILTERING

[Stage 2a] Correlation Pruning (threshold=0.85):
  High-correlation pair

In [5]:
# Stage 3: drop-column CV ranking (business scorer, avoids full-data RFECV leakage)
print("=" * 70)
print("STAGE 3: DROP-COLUMN CV RANKING (BUSINESS SCORER)")
print("=" * 70)

RANKING_CV_FOLDS = 5
ranked_stage2_features = rank_features_drop_column_cv(
    X_stage2,
    y,
    estimator_factory=get_classifier,
    cv=RANKING_CV_FOLDS,
)

print("\n✓ Drop-column ranking completed")
print(ranked_stage2_features.head(10).to_string(index=False))

STAGE 3: DROP-COLUMN CV RANKING (BUSINESS SCORER)

✓ Drop-column ranking completed
feature  cv_score_if_dropped  delta  order
var_389              -2497.0  187.0      1
var_308              -2496.0  188.0      2
 var_93              -2496.0  188.0      3
 var_87              -2492.0  192.0      4
var_254              -2490.0  194.0      5
var_365              -2488.0  196.0      6
var_420              -2487.0  197.0      7
var_463              -2487.0  197.0      8
var_448              -2485.0  199.0      9
var_190              -2483.0  201.0     10


In [6]:
ranked_stage2_features.to_csv(OUTPUTS_PATH / "feature_selection_results.csv", index=False)

## Extra Feature Selection Methods

New ranking methods on `X_stage2`:
- Mean Decrease in Impurity (Random Forest)
- Permutation Feature Importance
- Boruta (`boruta_py`)

In [7]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

X_stage2_df = X_stage2.copy()

rf_for_importance = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)
rf_for_importance.fit(X_stage2_df, y)

mdi_ranking = (
    pd.DataFrame({
        "feature": X_stage2_df.columns,
        "mdi_importance": rf_for_importance.feature_importances_,
    })
    .sort_values("mdi_importance", ascending=False)
    .reset_index(drop=True)
)
mdi_ranking["order"] = np.arange(1, len(mdi_ranking) + 1)

perm_result = permutation_importance(
    rf_for_importance,
    X_stage2_df,
    y,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
perm_ranking = (
    pd.DataFrame({
        "feature": X_stage2_df.columns,
        "perm_importance_mean": perm_result.importances_mean,
        "perm_importance_std": perm_result.importances_std,
    })
    .sort_values("perm_importance_mean", ascending=False)
    .reset_index(drop=True)
)
perm_ranking["order"] = np.arange(1, len(perm_ranking) + 1)


boruta_estimator = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)
boruta_selector = BorutaPy(
    estimator=boruta_estimator,
    n_estimators="auto",
    random_state=42,
    verbose=0,
)
boruta_selector.fit(X_stage2_df.values, y.values)

boruta_ranking = (
    pd.DataFrame({
        "feature": X_stage2_df.columns,
        "boruta_rank": boruta_selector.ranking_,
        "boruta_selected": boruta_selector.support_,
        "boruta_tentative": boruta_selector.support_weak_,
    })
    .sort_values(["boruta_rank", "feature"])
    .reset_index(drop=True)
)

print("Top 10 MDI:")
display(mdi_ranking.head(10))
print("Top 10 Permutation Importance:")
display(perm_ranking.head(10))
print("Top 10 Boruta:")
display(boruta_ranking.head(10))

Top 10 MDI:


,feature,mdi_importance,order
0,var_190,0.057306,1
1,var_254,0.056337,2
2,var_389,0.046890,3
3,var_308,0.044078,4
4,var_443,0.037738,5
5,var_54,0.037697,6
6,var_376,0.037415,7
7,var_220,0.036954,8
8,var_211,0.036892,9
9,var_463,0.036744,10


Top 10 Permutation Importance:


,feature,perm_importance_mean,perm_importance_std,order
0,var_190,0.04772,0.001840,1
1,var_254,0.04620,0.002464,2
2,var_308,0.00996,0.001073,3
3,var_389,0.00760,0.001073,4
4,var_54,0.00022,0.000108,5
5,var_320,0.00014,0.000156,6
6,var_211,0.00014,0.000128,7
7,var_376,0.00008,0.000098,8
8,var_392,0.00006,0.000092,9
9,var_126,0.00002,0.000060,10


Top 10 Boruta:


,feature,boruta_rank,boruta_selected,boruta_tentative
0,var_190,1,True,False
1,var_254,1,True,False
2,var_308,1,True,False
3,var_389,1,True,False
4,var_54,2,False,True
5,var_376,3,False,False
6,var_392,3,False,False
7,var_463,3,False,False
8,var_211,5,False,False
9,var_365,6,False,False


## Top-1 / Top-5 / Top-10 Comparison Across Algorithms

Check if methods choose different features in top lists.

In [ ]:
topk = (1, 5, 10)

algorithm_rankings = {
    "drop_column_cv": ranked_stage2_features["feature"].tolist(),
    "mdi": mdi_ranking["feature"].tolist(),
    "permutation": perm_ranking["feature"].tolist(),
}

algorithm_rankings["boruta"] = boruta_ranking["feature"].tolist()

for k in topk:
    print(f"\nTop-{k} features by method:")
    panel = pd.DataFrame({
        name: features[:k] + [None] * (k - len(features[:k]))
        for name, features in algorithm_rankings.items()
    })
    display(panel)


Top-1 features by method:


,drop_column_cv,mdi,permutation,boruta
0,var_389,var_190,var_190,var_190



Top-5 features by method:


,drop_column_cv,mdi,permutation,boruta
0,var_389,var_190,var_190,var_190
1,var_308,var_254,var_254,var_254
2,var_93,var_389,var_308,var_308
3,var_87,var_308,var_389,var_389
4,var_254,var_443,var_54,var_54



Top-10 features by method:


,drop_column_cv,mdi,permutation,boruta
0,var_389,var_190,var_190,var_190
1,var_308,var_254,var_254,var_254
2,var_93,var_389,var_308,var_308
3,var_87,var_308,var_389,var_389
4,var_254,var_443,var_54,var_54
5,var_365,var_54,var_320,var_376
6,var_420,var_376,var_211,var_392
7,var_463,var_220,var_376,var_463
8,var_448,var_211,var_392,var_211
9,var_190,var_463,var_126,var_365


In [ ]:
feature_sets_for_profit = {
    f"{method}_top{k}": ranked[:k] for method, ranked in algorithm_rankings.items() for k in topk
}

method_profit_scores = evaluate_feature_sets(
    X_stage2,
    y,
    feature_sets_for_profit,
    estimator_factory=get_classifier,
    cv=RANKING_CV_FOLDS,
)

method_profit_scores[["method", "k"]] = method_profit_scores["feature_set_name"].str.extract(
    r"^(.*)_top(\d+)$"
)
method_profit_scores["k"] = method_profit_scores["k"].astype(int)

In [15]:
profit_pivot_pen = method_profit_scores.pivot(
    index="method",
    columns="k",
    values="cv_score_mean",
)

profit_pivot_f1 = method_profit_scores.pivot(
    index="method",
    columns="k",
    values="f1_score",
)

print("Profit Pivot (penalty):")
display(profit_pivot_pen)
print("Profit Pivot (F1):")
display(profit_pivot_f1)

Profit Pivot (penalty):


k,1,5,10
method,,,
boruta,2268.0,1520.0,533.0
drop_column_cv,2301.0,1490.0,486.0
mdi,2268.0,1513.0,482.0
permutation,2268.0,1520.0,512.0


Profit Pivot (F1):


k,1,5,10
method,,,
boruta,0.664883,0.669704,0.671042
drop_column_cv,0.667868,0.666908,0.666538
mdi,0.664883,0.669152,0.666115
permutation,0.664883,0.669704,0.669015
